# 📈 CSV ➔ Insight Report Generator
## Automated Statistical Profiling, Anomaly Detection & Executive Markdown Briefing

**Pack ID:** `pack_0002` · **Category:** `ai_ops` · **Difficulty:** `intermediate` · **Version:** `1.1.0`

---
### 📋 Overview
Transforms tabular business and operational CSV data into statistical metrics and executive insight reports.


In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_PATH = Path('data/input.csv')
OUT_STATS = Path('outputs/statistics.json')
OUT_REPORT = Path('outputs/report.md')
OUT_STATS.parent.mkdir(parents=True, exist_ok=True)

print(f'Loading data from {INPUT_PATH}...')
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df)} rows, {len(df.columns)} columns.')


In [ ]:
# Statistical Analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
stats = {}
for col in numeric_cols:
    stats[col] = {
        'mean': round(float(df[col].mean()), 2),
        'median': round(float(df[col].median()), 2),
        'std': round(float(df[col].std()), 2),
        'min': float(df[col].min()),
        'max': float(df[col].max())
    }

plan_dist = df['plan'].value_counts().to_dict()
churn_rate = round(float(df['churned'].mean() * 100), 1)
total_mrr = float(df['mrr'].sum())

stats_payload = {
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'total_records': len(df),
    'total_mrr': total_mrr,
    'churn_rate_pct': churn_rate,
    'plan_distribution': plan_dist,
    'numeric_metrics': stats
}

with open(OUT_STATS, 'w', encoding='utf-8') as f:
    json.dump(stats_payload, f, indent=2)
print(f'✅ Written: {OUT_STATS}')


In [ ]:
# Generate Markdown Report
report_content = f'''# 📊 Executive CSV Insight Report

**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}  
**Total Records:** {len(df)}  
**Total MRR Analyzed:** ${total_mrr:,.2f}  
**Average Churn Rate:** {churn_rate}%  

## 1. Key Metrics Summary

| Metric | Mean | Median | Min | Max |
|---|---|---|---|---|
| **MRR** | ${stats['mrr']['mean']} | ${stats['mrr']['median']} | ${stats['mrr']['min']} | ${stats['mrr']['max']} |
| **NPS Score** | {stats['nps_score']['mean']} | {stats['nps_score']['median']} | {stats['nps_score']['min']} | {stats['nps_score']['max']} |

## 2. Customer Cohort Distribution

'''
for plan, count in plan_dist.items():
    report_content += f'- **{plan} Tier:** {count} accounts ({count/len(df)*100:.1f}%)\n'

report_content += '''
## 3. Strategic Recommendations
1. **Enterprise Retention:** Enterprise accounts generate over 80% of total MRR with zero churn. Expand enterprise sales coverage.
2. **Starter Tier Churn:** Starter accounts show higher churn (28%). Implement onboarding check-ins within the first 14 days.
'''

with open(OUT_REPORT, 'w', encoding='utf-8') as f:
    f.write(report_content)
print(f'✅ Written: {OUT_REPORT}')


In [ ]:
# Smoke Test
assert OUT_STATS.exists() and OUT_STATS.stat().st_size > 0
assert OUT_REPORT.exists() and OUT_REPORT.stat().st_size > 0
print('✅ Pack 0002 execution completed successfully.')
